# Experiments

## Setup: Import Libraries and Scripts

In [7]:
import pandas as pd
from IPython.display import display, HTML
import optimize_prompt as opt  # Full optimization script
import zero_shot_baseline as zsb  # Zero-shot baseline script

# Style for better table display
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)  # Show full content in cells

## Define Experiments

Add/edit experiments here. Each is a dict with:
- `'name'`: A label for the experiment.
- `'script'`: 'optimize' or 'zero_shot'.
- Other keys: Parameters for main() (e.g., generations, model_name).

In [8]:
# Run this cell to clear previous results
results = []

In [ ]:
experiments = [
    {
        'name': 'Zero-Shot Baseline (Default)',
        'script': 'zero_shot',
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'test_sample_size': 1000,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
    {
        'name': 'Full Optimization (Small)',
        'script': 'optimize',
        'generations': 30,
        'pop_size': 10,
        'train_sample_size': 50,
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
    {
        'name': 'Full Optimization (Small) - No Bandit',
        'script': 'optimize',
        'generations': 30,
        'pop_size': 10,
        'train_sample_size': 50,
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
]

## Run Experiments

This cell runs each experiment and collects results.

In [10]:
for exp in experiments:
    print(f"\n=== Running Experiment: {exp['name']} ===")
    script = exp.pop('script')  # Remove script key for passing to main
    name = exp.pop('name')  # Remove name for passing to main
    
    try:
        if script == 'optimize':
            result = opt.main(**exp)
        elif script == 'zero_shot':
            result = zsb.main(**exp)
        else:
            raise ValueError(f"Unknown script: {script}")
        
        # Flatten metrics for table
        metrics = result['test_metrics']
        flat_result = {
            'Experiment Name': name,
            'Script': script,
            **exp,  # Add back parameters
            'Best Instruction': result['best_instruction'],
            'Best Template': result['best_template'],
            'Sample Size': metrics['sample_size'],
            'Valid Predictions': metrics['valid_predictions'],
            'Total Predictions': metrics['total_predictions'],
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1 Micro': metrics['f1_micro'],
            'F1 Macro': metrics['f1_macro'],
            'Support (0/1)': f"{metrics['support'].get('0', 0)} / {metrics['support'].get('1', 0)}",
            'Unique y_true': ', '.join(metrics['unique_y_true']),
            'Unique y_pred': ', '.join(metrics['unique_y_pred']),
            'Detailed Report': metrics['detailed_report_string'],  # Full string for details
            'Full Classification Report (Dict)': metrics['classification_report']  # Raw dict if needed
        }
        results.append(flat_result)
    except Exception as e:
        print(f"Error in experiment '{name}': {e}")
        results.append({'Experiment Name': name, 'Error': str(e)})

# Create DataFrame
df_results = pd.DataFrame(results)


=== Running Experiment: Zero-Shot Baseline (Default) ===

Running zero-shot baseline on test set...


KeyboardInterrupt: 

## Display Results Table

Interactive table with all parameters and metrics. Scroll horizontally if needed.

In [ ]:
if not df_results.empty:
    # Style the table for better readability
    styled_df = df_results.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left')]}
    ]).background_gradient(cmap='viridis', subset=['F1 Macro'])
    display(HTML("<h3>Experiment Results</h3>"))
    display(styled_df)
else:
    print("No results to display.")

,Experiment Name,Script,model_name,test_sample_size,statutory_context_enabled,contract_context_enabled,Best Instruction,Best Template,Sample Size,Valid Predictions,Total Predictions,Accuracy,Precision,Recall,F1 Micro,F1 Macro,Support (0/1),Unique y_true,Unique y_pred,Detailed Report,Full Classification Report (Dict),generations,pop_size,train_sample_size,use_bandit_instr,use_bandit_template
0,Zero-Shot Baseline (Default),zero_shot,google/gemini-2.5-flash-lite-preview-06-17,100,True,True,Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.,Instruction: Clause: Statutory Context: Contract Context:,100,100,100,0.720000,0.166667,0.625000,0.720000,0.545159,92.0 / 8.0,"1, 0","1, 0",precision recall f1-score support 0 0.9571 0.7283 0.8272 92 1 0.1667 0.6250 0.2632 8 accuracy 0.7200 100 macro avg 0.5619 0.6766 0.5452 100 weighted avg 0.8939 0.7200 0.7820 100,"{'0': {'precision': 0.9571428571428572, 'recall': 0.7282608695652174, 'f1-score': 0.8271604938271605, 'support': 92.0}, '1': {'precision': 0.16666666666666666, 'recall': 0.625, 'f1-score': 0.2631578947368421, 'support': 8.0}, 'accuracy': 0.72, 'macro avg': {'precision': 0.5619047619047619, 'recall': 0.6766304347826086, 'f1-score': 0.5451591942820013, 'support': 100.0}, 'weighted avg': {'precision': 0.893904761904762, 'recall': 0.72, 'f1-score': 0.7820402858999351, 'support': 100.0}}",nan,nan,nan,nan,nan
1,Full Optimization (Small),optimize,google/gemini-2.5-flash-lite-preview-06-17,100,True,True,Carefully reread the clause and context before classifying. Classify the following clause from a Terms of Service contract as fair ('0') or unfair ('1') using the statutory context and contract context for better understanding. Respond only with '0' or '1'.,"Instruction: Clause: Statutory Context: Contract Context: Consider the provided and the . Based on the and , classify the clause as either fair ('0') or unfair ('1').",100,100,100,0.790000,0.227273,0.555556,0.790000,0.599160,91.0 / 9.0,"1, 0","1, 0",precision recall f1-score support 0 0.9487 0.8132 0.8757 91 1 0.2273 0.5556 0.3226 9 accuracy 0.7900 100 macro avg 0.5880 0.6844 0.5992 100 weighted avg 0.8838 0.7900 0.8260 100,"{'0': {'precision': 0.9487179487179487, 'recall': 0.8131868131868132, 'f1-score': 0.8757396449704142, 'support': 91.0}, '1': {'precision': 0.22727272727272727, 'recall': 0.5555555555555556, 'f1-score': 0.3225806451612903, 'support': 9.0}, 'accuracy': 0.79, 'macro avg': {'precision': 0.587995337995338, 'recall': 0.6843711843711844, 'f1-score': 0.5991601450658522, 'support': 100.0}, 'weighted avg': {'precision': 0.8837878787878788, 'recall': 0.79, 'f1-score': 0.8259553349875931, 'support': 100.0}}",10.000000,6.000000,10.000000,True,True
2,Full Optimization (Small) - No Bandit,optimize,google/gemini-2.5-flash-lite-preview-06-17,100,True,True,"Let's analyze step-by-step. Base your classification solely on legal facts and Directive 93/13, avoiding any personal opinions. Carefully reread the provided clause and its surrounding contract context. Compare the clause to definitions of unfairness under Directive 93/13, considering the statutory context and contract context. Conclude whether the clause is fair or unfair. Respond only with '0' for fair or '1' for unfair.",New Template: Response:,100,100,100,0.630000,0.200000,0.900000,0.630000,0.536050,90.0 / 10.0,"1, 0","1, 0",precision recall f1-score support 0 0.9818 0.6000 0.7448 90 1 0.2000 0.9000 0.3273 10 accuracy 0.6300 100 macro avg 0.5909 0.7500 0.5361 100 weighted avg 0.9036 0.6300 0.7031 100,"{'0': {'precision': 0.9818181818181818, 'recall': 0.6, 'f1-score': 0.7448275862068966, 'support': 90.0}, '1': {'precision': 0.2, 'recall': 0.9, 'f1-score': 0.32727272727272727, 'support': 10.0}, 'accuracy': 0.63, 'macro avg': {'precision': 0.5909090909090909, 'recall': 0.75, 'f1-score': 0.5360501567398119, 'support': 100.0}, 'weighted avg'